In [7]:
from notebooks.features.feature_extraction import load_all_features

features = load_all_features(filenames=['sense_start.csv'], version='oligo')
import pandas as pd

from notebooks.consts import OLIGO_CSV

unprocessed_data = pd.read_csv(OLIGO_CSV)

In [8]:
import pandas as pd

# Subset the features DataFrame to only include the merge key and the column you want
merged_df = pd.merge(
    features[['index_oligo', 'sense_start']],
    unprocessed_data,
    on='index_oligo'
)

In [9]:
from notebooks.preprocessing import process_oligo_data

data = process_oligo_data(merged_df)

------------------------------------------------------------
PROCESSING FILTERING REPORT
------------------------------------------------------------
Initial raw rows loaded: 158,725

[0. BASE FILTERING]
Unsupported chemistry (Mixmers/DNA/None): 20,040
Steric blocking (True) eliminated: 0
Multiple genes (';' present) eliminated: 0
Missing inhibition (NaN) eliminated: 0

[1. UNMAPPED SEQUENCES (sense_start == -1)]
Samples eliminated: 821

[2. COHORT FILTERING (>= 50 samples)]
Cohorts: 318 -> 242 (76 eliminated)
Samples: 137,864 -> 135,481 (2,383 eliminated)

[3. SPARSE CELL LINE FILTERING (>= 1000 samples)]
Cell Lines: 27 -> 16 (11 eliminated)
Samples: 135,481 -> 128,220 (7,261 eliminated)

FINAL DATASET: 128,220 ASOs

ELIMINATED GROUPS BREAKDOWN

[ELIMINATED UNMAPPED SAMPLES] - 821 samples across 12 genes:
  • KCNQ2: 460 samples
  • IGF2-AS: 69 samples
  • ARL14EPP1: 67 samples
  • RP11-739B23.1: 59 samples
  • MIR6801: 54 samples
  • RP11-823E8.3: 52 samples
  • F12: 21 samples
  • AD

In [10]:
import pandas as pd
from tauso.data.consts import CANONICAL_GENE, CELL_LINE

# Assuming you ran: data = preprocess_oligo_data()

print(f"Total ASOs in cleaned dataset: {len(data)}\n")

with pd.option_context('display.max_rows', None):
    # 1. Check the newly consolidated Cell Lines
    print("=== Cleaned Cell Lines ===")
    cell_counts = data[CELL_LINE].value_counts()
    print(f"Total Unique Cell Lines: {len(cell_counts)}\n")
    print(cell_counts)

    print("\n" + "="*50 + "\n")

    # 2. Check the Genes
    print("=== Target Genes ===")
    gene_counts = data[CANONICAL_GENE].value_counts()
    print(f"Total Unique Target Genes: {len(gene_counts)}\n")
    print(gene_counts)

    print("\n" + "="*50 + "\n")

    # 3. Check the Intersections (This will show us if the "Whales" are still isolated)
    print("=== Gene & Cell Line Intersections ===")
    combo_counts = data.groupby([CANONICAL_GENE, CELL_LINE]).size().sort_values(ascending=False)
    print(combo_counts)

Total ASOs in cleaned dataset: 128220

=== Cleaned Cell Lines ===
Total Unique Cell Lines: 16

Cell_line
A431                     41650
HepG2                    15948
SH-SY5Y                  15274
Hep3B                    12971
A549                      8116
Huh7                      5308
SK-MEL-28                 4830
THP-1                     4410
HepaRG                    4006
LNCaP                     3499
U251                      2741
T24                       2262
VCaP                      2222
HUVEC                     1921
iCell cardiomyocytes2     1715
K-562                     1347
Name: count, dtype: int64


=== Target Genes ===
Total Unique Target Genes: 159

Canonical Gene Name
F12         5238
PMP22       5189
DGAT2       5123
MSH3        4105
LRRK2       3663
ATXN1       3519
FOXP3       3499
PSD3        3268
ATXN3       3149
HSD17B13    3145
APP         3006
DNM2        2998
ANGPTL3     2956
HTT         2919
ATXN2       2869
MALAT1      2867
KCNT1       2860
SMAD7    

In [11]:
from tauso.data.consts import CELL_LINE

def split_by_cell_line(data):
    """
    Splits the data by holding out specific cell lines for the absolute test set.
    """
    # Define the cell lines to hold out completely
    test_cell_lines = ['A549', 'THP-1', 'HepaRG', 'LNCaP']

    # Create the split
    test_set = data[data[CELL_LINE].isin(test_cell_lines)].copy()
    train_set = data[~data[CELL_LINE].isin(test_cell_lines)].copy()

    # Print the summary
    print(f"--- Training Set ---")
    print(f"Total ASOs: {len(train_set)}")
    print(f"Unique Cell Lines: {train_set[CELL_LINE].nunique()}")
    print(f"Unique Genes: {train_set[CANONICAL_GENE].nunique()}\n")

    print(f"--- Absolute Test Set ---")
    print(f"Total ASOs: {len(test_set)}")
    print(f"Held-out Cell Lines: {', '.join(test_cell_lines)}")
    print(f"Unique Genes in Test: {test_set[CANONICAL_GENE].nunique()}")

    # Check for gene overlap (some genes might exist in both sets, which is fine,
    # but it's good to know exactly how many are strictly zero-shot)
    train_genes = set(train_set[CANONICAL_GENE])
    test_genes = set(test_set[CANONICAL_GENE])
    overlap = train_genes.intersection(test_genes)

    print(f"\nGenes appearing in BOTH sets: {len(overlap)}")
    print(f"Genes appearing ONLY in Test Set (Zero-Shot): {len(test_genes) - len(overlap)}")

    return train_set, test_set


# 2. Now run the Train/Test Split on the filtered data
# (Using the A549, THP-1, HepaRG, LNCaP holdout strategy we set up earlier)
train_data, test_data = split_by_cell_line(data)

--- Training Set ---
Total ASOs: 108189
Unique Cell Lines: 12
Unique Genes: 109

--- Absolute Test Set ---
Total ASOs: 20031
Held-out Cell Lines: A549, THP-1, HepaRG, LNCaP
Unique Genes in Test: 57

Genes appearing in BOTH sets: 7
Genes appearing ONLY in Test Set (Zero-Shot): 50


In [12]:
from pathlib import Path
from notebooks.consts import OLIGO_CSV

# 2. Dynamically determine the save paths
base_dir = Path(OLIGO_CSV).parent
train_path = base_dir / 'oligo_train.csv'
test_path = base_dir / 'oligo_test.csv'

# 3. Save the datasets
print(f"Saving training set to: {train_path}")
train_data.to_csv(train_path, index=False)

print(f"Saving absolute test set to: {test_path}")
test_data.to_csv(test_path, index=False)

print("Data successfully split and saved!")

Saving training set to: /home/michael/career/tauso_article/tauso_source2/notebooks/competitors/oligo_ai/oligo_train.csv
Saving absolute test set to: /home/michael/career/tauso_article/tauso_source2/notebooks/competitors/oligo_ai/oligo_test.csv
Data successfully split and saved!
